**Codigo de lectura de archivos CSV**

In [1]:
#Paso 1: Importar librerias
import pandas as pd
import numpy as np
import os

from google.colab import drive

In [2]:
# Paso 2: Montar Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


El primer problema que se tiene que resolver es que algunos datos presentan un formato de columnas tituladas para marcar los pixeles, además hay que unificar dichos datos en un arreglo de datos final.

In [3]:
# Paso 3: Lectura y adaptación de los archivos CSV de datos

# Definir rutas de entrada y salida
input_folder = '/content/drive/MyDrive/FotosIAproyecto/Datos_Proyecto1_IE0435_I-2026'
output_folder = '/content/drive/MyDrive/FotosIAproyecto/DatasetGrupal'

# Columnas esperadas por cada arreglo
arr_column = 128*128+1

data_set = []

# Búsqueda de archivos CSV
GroupData_csv = [archivo for archivo in os.listdir(input_folder) if archivo.lower().endswith('.csv')]

# Verificación de cantidad de archivos
print(f"Se encontraron {len(GroupData_csv)} archivos CSV en la carpeta de entrada.")

for archivo in GroupData_csv:
    # Ruta completa del archivo
    archive_path = os.path.join(input_folder, archivo)

    # Lectura y verificación del separador de columnas
    try:
      df = pd.read_csv(archive_path, header=None, sep=None, engine='python')

      if df.shape[1] == 1:
        # Verificación de que el separador corresponde a ;
        df = df.iloc[:, 0].astype(str).str.split(';', expand=True)

      # Eliminación de las etiquetas de píxeles en cada columna
      df = df.apply(pd.to_numeric, errors='coerce') # Asegura que los 1 y 0 se reconozcan como números
      df = df.dropna() # Esto elimina las filas que no sean numéricas

      # Verificación del número de columnas con 1 y 0
      df = df.astype(int)
      if df.shape[1] != arr_column:
          print(f"Advertencia: {archivo} tiene {df.shape[1]} columnas, no {arr_column}. Se omitió.")
          continue
          # Para asegurar que solo queden valores 0 y 1
      single_values = np.unique(df.values)
      if not set(single_values).issubset({0, 1}):
          print(f"Advertencia: {archivo} contiene valores distintos de 0 y 1: {single_values}. Se omitió.")
          continue
      data_set.append(df)
      print(f"{archivo} cargado correctamente. Filas agregadas: {df.shape[0]}")

    except Exception as e:
        print(f"Error procesando {archivo}: {e}")

#Paso 4: Unificación del dataset final
if data_set:
    datasetgrupal_total = pd.concat(data_set, ignore_index=True) # Variable renombrada

    # Define la ruta del archivo de salida usando output_folder
    output_file_path = os.path.join(output_folder, 'dataset_final_unificado.csv')
    datasetgrupal_total.to_csv(output_file_path, index=False, header=False)

    print("\n" + "-" * 50)
    print(f"Dataset creado: {datasetgrupal_total.shape}")
    print(f"Archivo guardado en: {output_file_path}")

else:
    print("\nNo se pudo generar el dataset final.")

Se encontraron 4 archivos CSV en la carpeta de entrada.
dataset_JacobGonzalez.csv cargado correctamente. Filas agregadas: 30
matriz_final.csv cargado correctamente. Filas agregadas: 30
dataset_JL.csv cargado correctamente. Filas agregadas: 112
datasetKDVJ.csv cargado correctamente. Filas agregadas: 374

--------------------------------------------------
Dataset creado: (546, 16385)
Archivo guardado en: /content/drive/MyDrive/FotosIAproyecto/DatasetGrupal/dataset_final_unificado.csv
